In [ ]:
# Install dependencies
!pip install torch torchvision --index-url https://pytorch.org
!pip install diffusers transformers accelerate safetensors pillow

Looking in indexes: https://pytorch.org
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 52.8 MB/s eta 0:00:00a 0:00:01


In [2]:
!nvidia-smi

Wed Aug 19 22:28:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive') # Access Drive folder

Mounted at /content/drive


In [4]:
# List everything in the drive folder
!ls -la /content/drive/MyDrive/generative_rockets/

total 5478
-rw------- 1 root root  191003 Aug 19 21:11 10.png
-rw------- 1 root root 2071496 Aug 19 21:02 1.png
-rw------- 1 root root  409059 Aug 19 21:02 2.png
-rw------- 1 root root  401250 Aug 19 21:03 3.png
-rw------- 1 root root  509367 Aug 19 21:06 4.png
-rw------- 1 root root  347432 Aug 19 21:07 5.png
-rw------- 1 root root  694790 Aug 19 21:07 6.png
-rw------- 1 root root  445919 Aug 19 21:08 7.png
-rw------- 1 root root  307428 Aug 19 21:09 8.png
-rw------- 1 root root  229561 Aug 19 21:10 9.png


In [5]:
import torch
from diffusers import StableDiffusionXLPipeline
from PIL import Image
import os

assert torch.cuda.is_available(), "CUDA is required to run SDXL locally."

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [6]:
# downloads model from Hugging Face cache
pipeline = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

In [13]:
# downloads the adapter subfolder weights
pipeline.load_ip_adapter(
    "h94/IP-Adapter", 
    subfolder="sdxl_models", 
    weight_name="ip-adapter_sdxl.bin"
)

There are modules in UNet2DConditionModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


In [15]:
# Set strength scale (0.5 to 0.8 balances image influence vs text prompt)
pipeline.set_ip_adapter_scale(0.6)

# Force the VAE decoder to match the float16 pipeline precision
pipeline.vae.to(dtype=torch.float16)

# Setting up memory saving configurations
pipeline.enable_freeu(s1=0.9, s2=0.2, b1=1.3, b2=1.4)
pipeline.unet.to(memory_format=torch.channels_last) 
pipeline.vae.enable_slicing()
pipeline.vae.enable_tiling()

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


In [18]:
image_dir = "/content/drive/MyDrive/generative_rockets/" 
image_paths = [os.path.join(image_dir, f) for f in sorted(os.listdir(image_dir)) if f.endswith(('png', 'jpg', 'jpeg'))][:10]
output_path = "generated_fusion_spaceship.png"

reference_images = []

for p in image_paths:
    img = Image.open(p)
    img_rgb = img.convert("RGB")
    reference_images.append(img_rgb)

prompt = (
    "A spaceship with a black background. The spaceship is human made"
    "combining structural design cues from the reference images, "
    "the spaceship is placed backwards, we should see the engines"
    "intricate metal armor plating, glowing blue ion engines, it has a recolector of dark martter in the front,"
    "the engines are in the back and the sides, 4k resolution"
)

print("Generating image...")

generator = torch.Generator(device="cpu").manual_seed(1340)
generated_image = pipeline(
    prompt=prompt,
    ip_adapter_image=[reference_images],
    num_inference_steps=40,
    generator=generator,
).images[0]


generated_image.save(output_path)
print(f"Saved generated spaceship to {output_path}")


Generating image...


KeyboardInterrupt: 